# UNIBO Case Study: International Citation Relationships

This notebook explores the international citation relationships associated with the University of Bologna (UNIBO), serving as a case study within the broader *Map of Italian Science* project.

The objective is to identify:

- Which countries most frequently cite UNIBO publications
- Which countries are most frequently cited by UNIBO publications
- Whether citation relationships are balanced or asymmetric
- The geographical distribution of incoming and outgoing citation flows

This exploratory analysis serves two purposes. First, it provides a detailed understanding of the citation geography of a single institution. Second, it supports the selection and refinement of visualization techniques that will later be scaled to the other five institutions included in the study, enabling a comparative analysis of citation patterns across institutions.

---
> **Notebook structure**
> 1. Setup & data loading
> 2. Incoming - Who cites UNIBO? 
> 3. Outgoing -  Whom does UNIBO cites?
> 4. Diverging bar chart (inbound vs outbound)
> 5. Choropleth maps
> 6. Asymmetry analysis
> 7. Summary of findings


## 1. Setup & Data Loading

The analysis relies on Pandas for data manipulation and Plotly for interactive visualizations. Data loading, cleaning, and normalization procedures are implemented through reusable functions contained in `src/data_utils.py`.

Country names are standardized during loading to resolve naming inconsistencies (e.g., *Russia* vs. *Russian Federation*), ensuring that citation counts are aggregated correctly before analysis.

In [29]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pycountry
from plotly.subplots import make_subplots
from pathlib import Path
import sys

# ── Paths & Environment ──
try:
    CURRENT_DIR = Path(__file__).resolve().parent
except NameError:
    CURRENT_DIR = Path.cwd()

# Add parent directory of data_viz to sys.path to allow importing from src
if CURRENT_DIR.name == "data_viz":
    sys.path.append(str(CURRENT_DIR.parent))
else:
    sys.path.append(str(CURRENT_DIR))

from src.data_utils import (
    BASE_PATH,
    INSTITUTIONS,
    INSTITUTION_LABELS,
    COUNTRY_NAMES,
    load_country_data,
    load_institution,
    load_all,
    pivot_directions,
    to_iso3
)

# ── Direction colours ──
DIR_COLORS = {"incoming": "#B7990D", "outgoing": "#320E3B"}

In [30]:
unibo_df = load_institution("UNIBO", exclude_self=True)
unibo_wide = pivot_directions(unibo_df)

unibo_wide.head(10)

,country_code,country_name,incoming_count,outgoing_count,total,difference,log2_ratio
212,US,United States,6800261.0,8894677.0,15694938.0,2094416.0,0.387352
68,FR,France,3956305.0,4043901.0,8000206.0,87596.0,0.031594
70,GB,United Kingdom,1986678.0,2461117.0,4447795.0,474439.0,0.308955
50,DE,Germany,1801165.0,1828283.0,3629448.0,27118.0,0.021559
42,CN,China,2167251.0,878920.0,3046171.0,-1288331.0,-1.302062
60,ES,Spain,1262588.0,1051307.0,2313895.0,-211281.0,-0.264200
102,JP,Japan,889572.0,842595.0,1732167.0,-46977.0,-0.078272
151,NL,The Netherlands,679657.0,800205.0,1479862.0,120548.0,0.235562
34,CA,Canada,676387.0,761331.0,1437718.0,84944.0,0.170675
11,AU,Australia,661316.0,670465.0,1331781.0,9149.0,0.019822


## 2. Incoming - Who cites UNIBO? 

In [38]:
unibo_df = load_institution("UNIBO", exclude_self=True)
unibo_wide = pivot_directions(unibo_df)

# incoming top-15
incoming_top15 = (unibo_df[unibo_df["direction"] == "incoming"]
                 .sort_values("count", ascending=False)
                 .head(15))

fig = px.bar(
    incoming_top15,
    x="count", y="country_name",
    orientation="h",
    title="Top 15 Countries Citing UNIBO (Incoming, excl. Italy)",
    labels={"count": "Citation count", "country_name": ""},
    color_discrete_sequence=[DIR_COLORS["incoming"]],
    template="plotly_white",
    height=520
)
fig.update_layout(title_x=0.5)
fig.show()

**Observations — UNIBO inbound**

- The distribution is **highly right-skewed**: the United States alone contributes millions of citations, far ahead of all other countries.
- France and China follow, together with United Kingdom and Germany: this suggests that UNIBO’s publications are primarily cited by countries with very large research ecosystems, high publication output and strong integration in international science. This was expected, but still worth mentioning.
- We decided to exclude Italy from the dataset, since keeping it would result in its self-citations heavily dominating the counts and completely obscuring all other international citation patterns. Its data would act as noise rather than significant signals.
- The long tail (many countries with tiny counts) has important implications for visualisation: a linear colour scale on a choropleth will flatten everything outside the US. Logarithmic scaling is strongly recommended (see §2d).

## 3. Outgoing -  Whom does UNIBO cites?

In [32]:
outgoing_top15 = (unibo_df[unibo_df["direction"] == "outgoing"]
                  .sort_values("count", ascending=False)
                  .head(15))

fig = px.bar(
    outgoing_top15,
    x="count", y="country_name",
    orientation="h",
    title="Top 15 Countries Cited by UNIBO (outgoing, excl. Italy)",
    labels={"count": "Citation count", "country_name": ""},
    color_discrete_sequence=[DIR_COLORS["outgoing"]],
    template="plotly_white",
    height=520
)
fig.update_layout(title_x=0.5)
fig.show()


**Observations — UNIBO outbound**

- The United States dominates outbound even more strongly than inbound, suggesting an **epistemic dependence** on US-centered science production.
- France is roughly symmetric in both directions — a sign of **reciprocal exchange**.
- China shows a notable asymmetry: it appears strongly in inbound (Chinese researchers cite UNIBO) but much less in outbound. Possible interpretations:
  - Western-centric citation practices
  - Language/publication ecosystem differences
  - Disciplinary composition of UNIBO's output

## 4. Diverging Bar Chart — Inbound vs Outbound

To understand whether a university is a net "producer" or "consumer" of citations with specific countries, we use diverging bar charts. Inbound citations (how much a country cites the institution) extend to the left, while outbound citations (how much the institution cites that country) extend to the right.

In [33]:
top15_wide = unibo_wide.head(15).copy()

# Define the order: ascending 
country_order = (top15_wide.sort_values("total", ascending=True)["country_name"].tolist())

# Build long format for diverging chart
incoming_long = top15_wide[["country_name", "incoming_count"]].copy()
incoming_long["direction"] = "incoming"
incoming_long["value"]     = -incoming_long["incoming_count"]
incoming_long["count"]     = incoming_long["incoming_count"]

outgoing_long = top15_wide[["country_name", "outgoing_count"]].copy()
outgoing_long["direction"] = "outgoing"
outgoing_long["value"]     = outgoing_long["outgoing_count"]
outgoing_long["count"]     = outgoing_long["outgoing_count"]

diverging_df = pd.concat([
    incoming_long[["country_name", "direction", "value", "count"]],
    outgoing_long[["country_name", "direction", "value", "count"]]
])

max_val = diverging_df["value"].abs().max()

fig = px.bar(
    diverging_df,
    x="value", y="country_name",
    color="direction",
    orientation="h",
    custom_data=["count", "direction"],
    color_discrete_map=DIR_COLORS,
    title="incoming vs outgoing Citation Relationships — UNIBO (Top 15)",
    labels={"value": "← incoming  |  outgoing →", "country_name": ""},
    template="plotly_white",
    height=560,
    category_orders={"country_name": country_order} 
)
fig.update_traces(
    hovertemplate="<b>%{y}</b><br>Citations: %{customdata[0]:,.0f}<br>Direction: %{customdata[1]}<extra></extra>"
)
fig.update_xaxes(range=[-max_val * 1.05, max_val * 1.05])
fig.update_layout(title_x=0.5, bargap=0.15, legend_title_text="")
fig.add_vline(x=0, line_width=1.5, line_color="gray")
fig.show()


**Interpretation**

The diverging chart makes **asymmetries immediately visible**.
- Countries where the two bars are roughly equal length (e.g. France, Germany) represent balanced bilateral exchange.
- Countries where outbound >> inbound (e.g. United States) reveal a **citation dependency**.
- Countries where inbound >> outbound (e.g. China) reveal an **asymmetric incoming influence**.

## 5. Choropleth Maps — Global Geographic Distribution (log scale)

While bar charts are excellent for isolating the top 15 partners, they ignore the "long tail" of global citations. Here, we project the citation volumes onto a world map to view the institution's total geographic footprint. 

**Methodological Note on Scaling:** Because citation data is hyper-skewed (spanning from tens of millions of citations for the US to single digits for smaller nations), a linear color scale would render the map visually blank. We apply a base-10 logarithmic scale (`log10`) to compress the range, revealing the nuanced geographic texture of the university's global reach.

In [34]:
# Merge with all countries (including zero-citation ones = NaN on map = grey)
# We work directly with the wide table

unibo_incoming_full  = (unibo_df[unibo_df["direction"] == "incoming"]
                       .copy())
unibo_outgoing_full = (unibo_df[unibo_df["direction"] == "outgoing"]
                       .copy())

# Add log-scaled count for better colour range
for df in [unibo_incoming_full, unibo_outgoing_full]:
    df["log_count"]    = np.log10(df["count"].clip(lower=1))
    df["country_iso3"] = df["country_code"].apply(to_iso3)

# Drop rows where conversion failed (e.g. XK for Kosovo, which has no ISO-3)
unibo_incoming_full  = unibo_incoming_full.dropna(subset=["country_iso3"])
unibo_outgoing_full = unibo_outgoing_full.dropna(subset=["country_iso3"])

fig_in = px.choropleth(
    unibo_incoming_full,
    locations="country_iso3",
    locationmode="ISO-3", 
    color="log_count",
    hover_name="country_name",
    hover_data={"count": ":,", "log_count": False},
    color_continuous_scale="YlOrBr",
    title="incoming Citations to UNIBO (log₁₀ scale, excl. Italy)",
    labels={"log_count": "log₁₀(citations)"},
    template="plotly_white",
    height=420
)
fig_in.update_layout(title_x=0.5)
fig_in.show()

fig_out = px.choropleth(
    unibo_outgoing_full,
    locations="country_iso3",
    locationmode="ISO-3",
    color="log_count",
    hover_name="country_name",
    hover_data={"count": ":,", "log_count": False},
    color_continuous_scale="Purples",
    title="outgoing Citations from UNIBO (log₁₀ scale, excl. Italy)",
    labels={"log_count": "log₁₀(citations)"},
    template="plotly_white",
    height=420
)
fig_out.update_layout(title_x=0.5)
fig_out.show()

**Note on log scaling:** The raw counts span several orders of magnitude (the US has tens of millions; many countries have fewer than 100). A linear colour scale makes the map monochromatic. `log₁₀` compresses the range and reveals the geographic texture. Hover on any country to see the real count.

While these choropleth maps do not introduce new quantitative variables beyond the previous bar charts, they provide a crucial spatial dimension. This geographic projection delivers immediate visual impact, making the macro-level dominance of the US, Western Europe, and key Asian countries instantly legible.

## 6. Asymmetry Analysis
### Which countries does UNIBO cite more than they cite it back?

The asymmetry map encodes the directionality of UNIBO's citation relationships rather than their raw absolute volume. For each included nation, the plotted metric is $\log_2(\text{outbound}/\text{inbound})$.
Under this logarithmic framework:
* Symmetry ($0$): Appears as an off-white/pale cream tone, indicating a perfectly balanced, reciprocal citation exchange.
* Outbound Bias (Positive values / Violet shades): Means UNIBO cites that country more than it receives citations in return. A value of $+1$ indicates UNIBO sends twice as many citations as it receives.
* Inbound Bias (Negative values / Yellow shades): Means that country cites UNIBO more than UNIBO cites them back. A value of $-1$ indicates UNIBO receives double the citations it outputs.

### Methodological Note on Filtering Statistical Noise
During initial exploratory visualization, this map suffered from small-value volatility. Nations with very low absolute citation footprints (e.g., fewer than 10 total citations) generated extreme ratios due to random statistical fluctuations, artificially painting large sections of the globe with misleadingly intense colors.

To resolve this issue, we implemented a noise floor filter threshold of 500 total citations. Any country failing to hit this baseline volume is masked out of the asymmetry scale and rendered as an uncolored light-grey background landmass. This mathematical constraint ensures that the remaining colored regions reflect robust, systematically significant citation flows rather than long-tail data noise.


This map should be read alongside the diverging bar charts, not as a substitute. The bar charts show which countries matter most by volume; the asymmetry map shows the directional balance across the entire world, including countries too small to appear in any top-N ranking.


In [35]:
# log2_ratio > 0 → UNIBO cites them more (outgoing bias)
# log2_ratio < 0 → they cite UNIBO more (incoming bias)

# ── Define a Minimum Citation Threshold ──
# Any country with fewer than 500 total citations will be treated as "insufficient data"
MIN_CITATIONS = 500

unibo_wide["country_iso3"] = unibo_wide["country_code"].apply(to_iso3)

# Drop rows where conversion failed (e.g. XK for Kosovo, which has no ISO-3)
unibo_wide = unibo_wide.dropna(subset=["country_iso3"])

# ── Filter the Data for the Color Scale ──
# We create a new column specifically for plotting. 
# If a country doesn't meet the threshold, we change its log2_ratio to NaN.
# Plotly automatically colors NaN rows as grey (background landcolor).
unibo_wide["filtered_log2_ratio"] = np.where(
    unibo_wide["total"] >= MIN_CITATIONS, 
    unibo_wide["log2_ratio"], 
    np.nan
)

custom_colorscale = [
    [0.0,  "#FFD500"],  # strong inbound bias
    [0.25, "#FFE760"],  # mild inbound
    [0.5,  "#F5F0E8"],  # balanced
    [0.75, "#6B3E7A"],  # mild outbound
    [1.0,  "#23022E"],  # strong outbound bias
]

fig_asym = px.choropleth(
    unibo_wide,
    locations="country_iso3",
    locationmode="ISO-3",
    color="filtered_log2_ratio",
    hover_name="country_name",
    hover_data={"incoming_count": ":,",
                "outgoing_count": ":,",
                "total": ":,",
                "log2_ratio": ":.2f",
                "filtered_log2_ratio": False},
    color_continuous_scale=custom_colorscale,
    color_continuous_midpoint=0,
    range_color=[-3, 3],
    title="Citation Asymmetry — UNIBO",
    labels={"filtered_log2_ratio": "Outgoing ↔ Incoming"},
    template="plotly_white",
    height=440
)
fig_asym.update_layout(
    title_x=0.5,
    coloraxis_colorbar=dict(
        title="log₂(out/in)"
    )
)

fig_asym.show()

## 7. Findings: global asymmetry in citation flows

With the low-volume noise successfully filtered out, the global map displays a prominent, unmistakable yellow topology across Latin America, Africa, the Middle East, and much of Asia. This establishes a major structural insight that raw volume bar charts tend to hide: across the vast majority of the global academic landscape, UNIBO behaves as a net knowledge producer and citation receiver.

- The **United States** stands out as the primary **exception**, shifting distinctly into violet tones. This confirms our diverging bar chart analysis: UNIBO's outbound reliance on American publication venues is so massive that it easily overpowers the millions of incoming citations it receives from US researchers. This visually captures the structural, central gravity that Anglophone journals exert on European research practices. 

- **Northern Europe** (alongside Australia and Canada) occupy a balanced middle ground. Countries in this region appear as pale cream or near-white, indicating a reciprocal exchange of citations. This equilibrium is consistent with deep integration within shared European research infrastructures and long-standing co-authorship networks.

- Major research powerhouses like **China** and **India** stand out in clear, unmasked yellow. Because their citation volumes are well past the 500-citation threshold, this imbalance is highly meaningful. It explicitly shows an asymmetric relationship where Asian research ecosystems consume and build upon UNIBO’s scholarly output at a pace that outstrips how frequently Italian researchers cite those regions back.

#### Overall interpretation
The filtered asymmetry map reframes our view of the Map of Italian Science. While UNIBO exhibits a clear citation dependency on a selective tier of elite Anglophone and Western scientific epicenters (the violet regions), this pattern is concentrated within a very tight circle of high-volume partners.

On a truly global scale, the University of Bologna acts as a vital **net exporter of knowledge**. It bridges the gap in the global citation economy—operating within a tier that relies on hyper-centralized scientific production from the United States, while simultaneously serving as an **essential foundational reference** for expanding research communities throughout Asia and the Global South.